# Mooring Field Detection — Cape Cod + Florida (Kaggle GPU)

Detects on pre-fetched tiles. No Google fetch. No Groq.

## Settings
1. **GPU T4** + **Internet On** (Restart session after enabling GPU)
2. **Add input** → your dataset with Cape Cod / FL payloads
3. **Run All**

Works if the dataset contains either:
- many `kaggle_scan_*.zip` files, **or**
- extracted folders each with `candidates.kml` + `imagery/scan/` (+ `weights/`)

After success: download `scan_out/mooring_fields.db` →  
`python -m mooring_fields.cli import-scan --from-db <db> --all`

In [ ]:
# Cell 1 — sparse clone + install
import subprocess, sys, shutil, os
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
URL = "https://github.com/IshanKasam/MooringFieldDetection.git"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", URL, str(REPO)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO), "sparse-checkout", "set", "src", "config", "pyproject.toml", "README.md"],
    check=True,
)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "python-dotenv"],
    check=True,
)

import torch
from mooring_fields.runtime import cuda_available

print("cuda:", cuda_available(), torch.cuda.get_device_name(0) if cuda_available() else None)
print("inputs:", sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else None)
assert cuda_available(), "Settings → Accelerator → GPU T4 → Restart session, then re-run"

In [ ]:
# Cell 2 — discover payloads (zips AND/OR extracted folders)
# Kaggle often auto-extracts uploaded .zip files, so .zip may be gone.
from pathlib import Path

INPUT = Path("/kaggle/input")
assert INPUT.exists() and any(INPUT.iterdir()), "Add input → attach your dataset"

print("=== /kaggle/input layout (depth≤3) ===")
for p in sorted(INPUT.rglob("*")):
    rel = p.relative_to(INPUT)
    if len(rel.parts) > 3:
        continue
    kind = "dir" if p.is_dir() else "file"
    print(f"  [{kind}] {rel}")

zip_paths = sorted(INPUT.rglob("kaggle_scan_*.zip"))
if not zip_paths:
    zip_paths = sorted(
        z for z in INPUT.rglob("*.zip")
        if z.name != "kaggle_scan_payload.zip" or True
    )
names = {z.name for z in zip_paths}
if "kaggle_scan_CapeCod.zip" in names and "kaggle_scan_payload.zip" in names:
    zip_paths = [z for z in zip_paths if z.name != "kaggle_scan_payload.zip"]

# Extracted region dirs: any directory that contains candidates.kml
kml_dirs = []
seen = set()
for kml in sorted(INPUT.rglob("candidates.kml")):
    d = kml.parent.resolve()
    if d in seen:
        continue
    seen.add(d)
    scan_img = d / "imagery" / "scan"
    png_n = len(list(scan_img.glob("*.png"))) if scan_img.is_dir() else 0
    kml_dirs.append((d, png_n))

print("\nzip files found:", len(zip_paths))
for z in zip_paths:
    print(" ", z, f"({z.stat().st_size/1e6:.0f} MB)")
print("extracted candidates.kml dirs found:", len(kml_dirs))
for d, png_n in kml_dirs:
    print(f"  {d}  pngs={png_n}")

assert zip_paths or kml_dirs, (
    "No kaggle_scan_*.zip and no candidates.kml under /kaggle/input. "
    "Re-check that the dataset is attached (right sidebar → Input)."
)

if not zip_paths and len(kml_dirs) == 1:
    print(
        "\nWARNING: Only ONE extracted region found. "
        "If you uploaded many zips, Kaggle likely extracted them into the same "
        "paths and they overwrote each other. You will only get that one region. "
        "For all regions: re-upload with each zip kept as a .zip file, or put "
        "each extract in its own subfolder (CapeCod/, FL_tampa_sw_p0/, …)."
    )

In [ ]:
# Cell 3 — GPU detect every zip and/or every extracted region → one DB
import json, os, sys, shutil, zipfile
from pathlib import Path

REPO = Path("/kaggle/working/MooringFieldDetection")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from mooring_fields.cli import scan_cmd
from mooring_fields.kaggle_scan import materialize_kaggle_scan_input

out_dir = Path("/kaggle/working/scan_out")
out_dir.mkdir(parents=True, exist_ok=True)
db_path = out_dir / "mooring_fields.db"
extract_root = Path("/kaggle/working/payload_one")

# Build work list: (label, path_or_zip, kind)
jobs = []
for z in zip_paths:
    jobs.append((z.stem, z, "zip"))
if not zip_paths:
    for d, _png in kml_dirs:
        label = d.name if d.name not in ("input", ".") else d.parent.name
        # Prefer a meaningful folder name from path parts
        parts = d.relative_to(INPUT).parts if INPUT in d.parents or d == INPUT else (label,)
        label = "_".join(parts[:3]) if parts else label
        jobs.append((label, d, "dir"))

assert jobs, "Nothing to run"
print(f"Will run {len(jobs)} job(s)")

reports = []
for i, (label, src, kind) in enumerate(jobs, 1):
    safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in label)[:80]
    print(f"\n======== [{i}/{len(jobs)}] {safe} ({kind}) ========")

    if kind == "zip":
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True)
        with zipfile.ZipFile(src) as zf:
            zf.extractall(extract_root)
        payload_dir = extract_root
        if not (payload_dir / "candidates.kml").is_file():
            found = list(payload_dir.rglob("candidates.kml"))
            assert found, f"No candidates.kml in {src.name}"
            payload_dir = found[0].parent
    else:
        payload_dir = Path(src)

    layout = materialize_kaggle_scan_input(
        payload_dir,
        work_dir=Path("/kaggle/working/scan_run") / safe,
    )
    print(json.dumps({k: layout[k] for k in ("png_count", "weights")}, indent=2))
    assert layout["png_count"] > 0, layout
    assert layout["weights"], f"{safe}: missing weights/best.pt — zip must include weights/"

    scan_cmd([
        "--kml", layout["kml"],
        "--skip-fetch",
        "--imagery-dir", layout["imagery_dir"],
        "--weights", layout["weights"],
        "--db", str(db_path),
        "--output-dir", str(out_dir / safe),
    ])
    reports.append({"label": safe, "png_count": layout["png_count"], "kind": kind})

    if kind == "zip":
        shutil.rmtree(extract_root, ignore_errors=True)
    shutil.rmtree(Path("/kaggle/working/scan_run") / safe, ignore_errors=True)

assert db_path.is_file()
print("\n========== SUCCESS ==========")
print("Download:", db_path)
for r in reports:
    print(" ", r)
print("PC: python -m mooring_fields.cli import-scan --from-db <db> --all")
shutil.rmtree(REPO, ignore_errors=True)

## After

1. Download `scan_out/mooring_fields.db`
2. `python -m mooring_fields.cli import-scan --from-db ~\Downloads\mooring_fields.db --all`
3. Hard-refresh the map

If Cell 2 warned that only **one** extracted region exists, re-create the dataset so each region stays separate (keep `.zip` files, or one subfolder per region).